In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

def make_downloader(name, data_dir, batch_size=128, image_size=32):
  if name=='cifar10':
    train_transform=transforms.Compose([
        transforms.Resize((image_size,image_size)),
        transforms.RandomCrop(image_size, padding=4),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
      ])
    val_transforms=transforms.Compose([
        transforms.Resize((image_size,image_size)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    ])

    test_transforms=transforms.Compose([
        transforms.Resize((image_size,image_size)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
        ])


    full_train_dataset      = datasets.CIFAR10(root=data_dir, train=True,  download=False, transform=train_transform)
    full_train_dataset_val = datasets.CIFAR10(root=data_dir, train=True,  download=False, transform=val_transforms)
    full_test_dataset       = datasets.CIFAR10(root=data_dir, train=False, download=False, transform=test_transforms)


    train_size=int(0.9*(len(full_train_dataset)))
    val_size=len(full_train_dataset)-train_size


    genrator=torch.Generator().manual_seed(42)
    train_indices,val_indices=random_split(range(len(full_train_dataset)),[train_size,val_size],generator=genrator)

    train_dataset=torch.utils.data.Subset(full_train_dataset,train_indices.indices)
    val_dataset=torch.utils.data.Subset(full_train_dataset_val,val_indices.indices)
    test_dataset=full_test_dataset

    train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
    val_loader=DataLoader(val_dataset,batch_size=batch_size,shuffle=False)
    test_loader=DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

    return train_loader,val_loader,test_loader,10
  else:
    raise ValueError(f'Invalid dataset {name}')


train_loader,val_loader,test_loader,num_classes=make_downloader('cifar10',data_dir="/content/drive/MyDrive/Datasets")
print(train_loader)
print(val_loader)
print(test_loader)
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)


In [ ]:
# model.py
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights,vit_b_16,ViT_B_16_Weights
from torchvision.models.detection import fasterrcnn_resnet50_fpn,FasterRCNN_ResNet50_FPN_Weights
import torch
class CustomCNN(nn.Module):
  def __init__(self,num_classes=10):
    super().__init__()

    self.features=nn.Sequential(
        nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(64,64,kernel_size=3,stride=1,padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )

    self.classifier=nn.Sequential(
        nn.Linear(64*16*16,512),
        nn.ReLU(inplace=True),
        nn.Linear(512,num_classes)
    )

  def forward(self,x):
    x=self.features(x)
    x=x.view(x.size(0),-1)
    x=self.classifier(x)
    return x


class FasterRCNNBackboneClassifier(nn.Module):
  def __init__(self,num_classes=10):
    super().__init__()
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    self.backbone=fasterrcnn_resnet50_fpn(weights=weights).backbone
    self.classifier=nn.Sequential(
        nn.Linear(256,512),
        nn.ReLU(inplace=True),
        nn.Linear(512,num_classes)
    )


  def forward(self,x):
      features=self.backbone(x)
      choosen=features["0"]
      pool=nn.functional.adaptive_avg_pool2d(choosen,(1,1))
      flat=pool.view(pool.size(0),-1)
      return self.classifier(flat)



def make_model(arch,num_classes,pretrained=True):
  if arch=="custom_cnn":
     return CustomCNN(num_classes)
  elif arch=="convnext_tiny":
    weights=ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
    model=convnext_tiny(weights=weights)
    in_features=model.classifier[2].in_features
    model.classifier[2]=nn.Linear(in_features,num_classes)
    return model
  elif arch=="vit_b_16":
    weights=ViT_B_16_Weights.DEFAULT if pretrained else None
    model=vit_b_16(weights=weights)
    in_features=model.heads[0].in_features
    model.heads[0]=nn.Linear(in_features,num_classes)
    return model
  elif arch=="faster_rcnn_backbone":
    return FasterRCNNBackboneClassifier(num_classes)
  else:
     raise ValueError("Invalid architchture")

model=make_model("faster_rcnn_backbone",num_classes=10)
dummy=torch.randn(4,3,224,224)
print(model(dummy).shape)



In [ ]:
# from torchvision.models.detection import fasterrcnn_resnet50_fpn,FasterRCNN_ResNet50_FPN_Weights

# weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
# detector=fasterrcnn_resnet50_fpn(weights=weights)
# backbone=detector.backbone

# dummy=torch.randn(4,3,224,224)
# output=backbone(dummy)

# print(type(output))
# print(output.keys())
# for key, value in output.items():
#     print(key, value.shape)

In [ ]:
# utils/metrics.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay,confusion_matrix

def compute_metrics(y_true, y_pred, class_names):
  labels = list(range(len(class_names)))
  report = classification_report(y_true,y_pred, labels=labels, target_names=class_names, output_dict=True)
  per_class_df= pd.DataFrame(report).transpose().loc[class_names]
  global_df=pd.DataFrame(report).transpose().loc[["macro avg", "weighted avg"]]
  return per_class_df,global_df

def save_confusion_matrix(y_true, y_pred, class_names, save_path):
  labels = list(range(len(class_names)))
  cm=confusion_matrix(y_true, y_pred, labels=labels)
  display=ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
  display.plot(xticks_rotation=45)
  plt.tight_layout()
  plt.savefig(save_path)
  plt.close()

CIFAR10_CLASSES = [
    "airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"
    ]
fake_true=np.random.randint(0,10,200)
fake_pred=np.random.randint(0,10,200)
per_class_df, global_df= compute_metrics(fake_true,fake_pred,CIFAR10_CLASSES)
print(per_class_df)
print(global_df)

save_confusion_matrix(fake_true, fake_pred, CIFAR10_CLASSES, "test_cm.png")

# Train.py

Stage 1 — the core training step



In [ ]:
# train.py
def train(model, loader, optimizer, criterion, device):
  model.train()
  total_loss=0
  for images,labels in loader:
    images,labels=images.to(device),labels.to(device)
    # your 5 lines here: zero_grad, forward, loss, backward, step
    optimizer.zero_grad()
    logits=model(images)
    loss= criterion(logits,labels)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss/len(loader)


Stage 2 — validation function

In [ ]:
def evaluate(model, loader,criterion, device):
  model.eval()
  total_loss=0
  all_pred=[]
  all_labels=[]

  with torch.no_grad():
    for images,labels in loader:
      images,labels=images.to(device),labels.to(device)
      logits=model(images)
      loss=criterion(logits,labels)
      _,preds=torch.max(logits,1)
      all_pred.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())
      total_loss+=loss.item()

    avg_loss=total_loss/len(loader)
    return avg_loss,all_pred,all_labels


Stage3: End to end smog test

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader,val_loader,test_loader,num_classes=make_downloader('cifar10',data_dir="/content/drive/MyDrive/Datasets", image_size=32)
model=make_model("custom_cnn",num_classes=num_classes).to(device)
optimizer= torch.optim.AdamW(model.parameters(),lr=0.001)
criterion=torch.nn.CrossEntropyLoss()

train_loss=train(model,train_loader,optimizer,criterion,device)
print(train_loss)
val_loss,val_preds,val_labels=evaluate(model,val_loader,criterion,device)
print("train_loss:", train_loss)
print("val_loss:", val_loss)
print("num val preds:", len(val_preds))

Stage 4 — build the config-driven

In [ ]:
import random
import numpy as np
import torch

def setup_training(model, lr=0.0001, use_weight_decay=True, use_scheduler=True, freeze_backbone=False,seed=42, epochs=10):
  random.seed(42)
  np.random.seed(42)
  torch.manual_seed(42)
  torch.cuda.manual_seed_all(42)

  if freeze_backbone and hasattr(model, "backbone"):
    for param in model.backbone.parameters():
      param.requires_grad=False

  weight_decay=0.0001 if use_weight_decay else 0.0
  optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=weight_decay)
  scheduler= torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs) if use_scheduler else None

  return optimizer, scheduler

def train_epochs(model, loader, optimizer, criterion, device, use_amp=False, use_grid_clip=False, max_norm=1.0):
  model.train()
  total_loss=0
  scaler= torch.cuda.amp.GradScaler(enabled=use_amp)

  for images,labels in loader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    with torch.cuda.amp.autocast(enabled=use_amp):
      logits=model(images)
      loss=criterion(logits,labels)
    scaler.scale(loss).backward()

    if use_grid_clip:
      scaler.unscale_(optimizer)
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

    scaler.step(optimizer)
    scaler.update()

    total_loss+=loss.item()

  return total_loss/len(loader)


config={
    "lr":0.001,
    "use_weight_decay":True,
    "use_scheduler":True,
    "use_amp":False,
    "use_grid_clip":False,
    "freeze_backbone":False,
    "seed":42,
    "epochs":1,
}

# Filter config for setup_training arguments
setup_training_args = {k: config[k] for k in ["lr", "use_weight_decay", "use_scheduler", "freeze_backbone", "seed", "epochs"] if k in config}

# Filter config for train_epochs arguments
train_epochs_args = {k: config[k] for k in ["use_amp", "use_grid_clip"] if k in config}

model=make_model("custom_cnn",num_classes=10).to(device)
optimizer, scheduler=setup_training(model, **setup_training_args)
criterion=torch.nn.CrossEntropyLoss()
for epoch in range(config["epochs"]):
  train_loss=train_epochs(model,train_loader,optimizer,criterion,device, **train_epochs_args)
  if scheduler is not None:
    scheduler.step()

  val_loss,val_preds,val_labels=evaluate(model,val_loader,criterion,device)
  print(f"epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

def save_checkpoint(model, optimizer, epoch, val_loss, path):
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": epoch,
        "val_loss": val_loss,
    }, path)


best_val_loss = float("inf")
for epoch in range(config["epochs"]):
  train_loss=train_epochs(model,train_loader,optimizer,criterion,device, **train_epochs_args)
  if scheduler is not None:
    scheduler.step()

  val_loss,val_preds,val_labels=evaluate(model,val_loader,criterion,device)
  print(f"epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

  if val_loss < best_val_loss:
    best_val_loss = val_loss
    save_checkpoint(model, optimizer, epoch, val_loss, "best_model.pt")
    print(f"  saved new best checkpoint (val_loss={val_loss:.4f})")

In [ ]:
import os

os.makedirs("part_1_results", exist_ok=True)

# evaluate on the TEST set (not val) for final reported numbers
test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

per_class_df, global_df = compute_metrics(test_labels, test_preds, CIFAR10_CLASSES)

arch_name = "custom_cnn"
dataset_name = "cifar10"

summary = f"Experiment: {arch_name} | Dataset: {dataset_name}\n"
summary += per_class_df.to_string()
summary += "\n\n"
summary += global_df.to_string()
summary += "\n\n"

with open("part_1_results/summary.txt", "w") as f:
    f.write(summary)

save_confusion_matrix(test_labels, test_preds, CIFAR10_CLASSES, f"part_1_results/cn_{arch_name}_{dataset_name}.png")

print(summary)

Export ONNX

In [ ]:
model.eval()
dummy_input = torch.randn(1, 3, 32, 32).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "custom_cnn_cifar10_fixed.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamo=False
)

print("exported fixed-batch model")

# Sanity check: does ONNX give the same output as PyTorch

In [ ]:
!pip install onnxruntime

In [ ]:

import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("custom_cnn_cifar10.onnx")

test_input = torch.randn(1, 3, 32, 32).to(device)

# PyTorch output
model.eval()
with torch.no_grad():
    torch_output = model(test_input).cpu().numpy()

# ONNX output
onnx_output = session.run(None, {"input": test_input.cpu().numpy()})[0]

print("PyTorch output:", torch_output)
print("ONNX output:", onnx_output)
print("Max difference:", np.abs(torch_output - onnx_output).max())

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="custom_cnn_cifar10_fixed.onnx",
    model_output="custom_cnn_cifar10_dynamic_int8.onnx",
    weight_type=QuantType.QInt8
)

print("dynamic quantization done")

import os
fp32_size = os.path.getsize("custom_cnn_cifar10_fixed.onnx") / 1024
int8_size = os.path.getsize("custom_cnn_cifar10_dynamic_int8.onnx") / 1024
print(f"FP32 size: {fp32_size:.1f} KB")
print(f"Dynamic INT8 size: {int8_size:.1f} KB")

# Step 1: Run inference with the INT8 ONNX model and compare to PyTorch

In [ ]:
import onnxruntime as ort
import numpy as np

int8_session = ort.InferenceSession("custom_cnn_cifar10_dynamic_int8.onnx")

test_input = torch.randn(1, 3, 32, 32).to(device)

model.eval()
with torch.no_grad():
    torch_output = model(test_input).cpu().numpy()

int8_output = int8_session.run(None, {"input": test_input.cpu().numpy()})[0]

print("PyTorch output:", torch_output)
print("INT8 output:", int8_output)
print("Max difference:", np.abs(torch_output - int8_output).max())

Step 2 — Static quantization (harder one, needs calibration data)

In [ ]:
from onnxruntime.quantization import CalibrationDataReader, quantize_static, QuantType, QuantFormat

class CIFAR10CalibrationReader(CalibrationDataReader):
    def __init__(self, loader, num_batches=10):
        self.iterator = iter(loader)
        self.num_batches = num_batches
        self.count = 0

    def get_next(self):
        if self.count >= self.num_batches:
            return None
        try:
            images, _ = next(self.iterator)
        except StopIteration:
            return None
        self.count += 1
        return {"input": images[:1].numpy()}  # one image at a time, matches our fixed batch=1 export

calibration_reader = CIFAR10CalibrationReader(val_loader, num_batches=20)

quantize_static(
    model_input="custom_cnn_cifar10_fixed.onnx",
    model_output="custom_cnn_cifar10_static_int8.onnx",
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,
    weight_type=QuantType.QInt8,
)

print("static quantization done")

static_size = os.path.getsize("custom_cnn_cifar10_static_int8.onnx") / 1024
print(f"Static INT8 size: {static_size:.1f} KB")

In [ ]:
from torch.utils.data import Subset

small_test_dataset = Subset(test_loader.dataset, range(500))  # 500 images instead of 10,000
small_test_loader = DataLoader(small_test_dataset, batch_size=128, shuffle=False)

print(len(small_test_loader.dataset))

In [ ]:
def evaluate_onnx(session, loader, num_classes):
    all_preds = []
    all_labels = []
    for images, labels in loader:
        for i in range(images.size(0)):
            single_image = images[i:i+1].numpy()  # keep batch dim = 1, matches fixed export
            output = session.run(None, {"input": single_image})[0]
            pred = output.argmax(axis=1)[0]
            all_preds.append(pred)
            all_labels.append(labels[i].item())
    return all_preds, all_labels
fp32_session = ort.InferenceSession("custom_cnn_cifar10_fixed.onnx")
# Initialize static_session
static_session = ort.InferenceSession("custom_cnn_cifar10_static_int8.onnx")

fp32_preds, fp32_labels = evaluate_onnx(fp32_session, small_test_loader, num_classes)
dynamic_preds, dynamic_labels = evaluate_onnx(int8_session, small_test_loader, num_classes)
static_preds, static_labels = evaluate_onnx(static_session, small_test_loader, num_classes)

print("done")

In [ ]:
import os

os.makedirs("part_2_results", exist_ok=True)

def get_summary_metrics(y_true, y_pred, class_names):
    per_class_df, global_df = compute_metrics(y_true, y_pred, class_names)
    accuracy = (np.array(y_true) == np.array(y_pred)).mean()
    macro_f1 = global_df.loc["macro avg", "f1-score"]
    weighted_f1 = global_df.loc["weighted avg", "f1-score"]
    return accuracy, macro_f1, weighted_f1

fp32_acc, fp32_macro_f1, fp32_weighted_f1 = get_summary_metrics(fp32_labels, fp32_preds, CIFAR10_CLASSES)
dynamic_acc, dynamic_macro_f1, dynamic_weighted_f1 = get_summary_metrics(dynamic_labels, dynamic_preds, CIFAR10_CLASSES)
static_acc, static_macro_f1, static_weighted_f1 = get_summary_metrics(static_labels, static_preds, CIFAR10_CLASSES)

comparison_df = pd.DataFrame({
    "Model Artifact": ["PyTorch FP32", "ONNX INT8 Dynamic", "ONNX INT8 Static"],
    "Accuracy": [fp32_acc, dynamic_acc, static_acc],
    "Macro F1": [fp32_macro_f1, dynamic_macro_f1, static_macro_f1],
    "Weighted F1": [fp32_weighted_f1, dynamic_weighted_f1, static_weighted_f1],
})

summary_text = "Experiment: custom_cnn | Dataset: cifar10\n\n"
summary_text += comparison_df.to_string(index=False)

with open("part_2_results/summary.txt", "w") as f:
    f.write(summary_text)

print(summary_text)

In [ ]:
CIFAR10_CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

def evaluate_pytorch_checkpoint(ckpt_path, arch, loader, device):
    eval_model = make_model(arch, num_classes=10).to(device)
    checkpoint = torch.load(ckpt_path, map_location=device)
    eval_model.load_state_dict(checkpoint["model_state"])
    eval_model.eval()

    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = eval_model(images)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return all_preds, all_labels

# run it on your saved checkpoint
eval_preds, eval_labels = evaluate_pytorch_checkpoint("best_model.pt", "custom_cnn", small_test_loader, device)

per_class_df, global_df = compute_metrics(eval_labels, eval_preds, CIFAR10_CLASSES)
print(per_class_df)
print(global_df)